# Colab Setup

Run this notebook **once per Colab session** before opening any stage notebook.

What it does:
1. Clones (or pulls) the repo to `/content/ctls`
2. Mounts Google Drive and creates persistent directories for data and checkpoints
3. Symlinks `data/raw` and `experiments/` → Drive so nothing is lost on runtime restart
4. Installs dependencies

After this cell completes, open any stage notebook from the left file browser under `/content/ctls/notebooks/`.

## Step 1 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('Drive mounted at /content/drive')

## Step 2 — Clone / Update the Repo

Edit `REPO_URL` to point to your GitHub repo.

In [ ]:
import os

REPO_URL = 'https://github.com/YOUR_USERNAME/model_interpretability.git'  # <-- edit this
REPO_DIR = '/content/ctls'

if os.path.exists(REPO_DIR):
    print('Repo already cloned — pulling latest changes...')
    !git -C {REPO_DIR} pull
else:
    print('Cloning repo...')
    !git clone {REPO_URL} {REPO_DIR}

print('\nRepo ready at', REPO_DIR)

## Step 3 — Create Persistent Directories on Drive and Symlink Them

In [ ]:
DRIVE_BASE = '/content/drive/MyDrive/ctls'

# Persistent directories on Drive
drive_data    = f'{DRIVE_BASE}/data/raw'
drive_exp     = f'{DRIVE_BASE}/experiments'

os.makedirs(drive_data, exist_ok=True)
os.makedirs(drive_exp,  exist_ok=True)

# Ensure repo-side parent dirs exist
os.makedirs(f'{REPO_DIR}/data', exist_ok=True)

# Create symlinks (remove stale symlinks first if re-running)
for link, target in [
    (f'{REPO_DIR}/data/raw',     drive_data),
    (f'{REPO_DIR}/experiments',  drive_exp),
]:
    if os.path.islink(link):
        os.unlink(link)
    os.symlink(target, link)
    print(f'  {link}  ->  {target}')

print('\nSymlinks created. Data and checkpoints will persist on Drive across sessions.')

## Step 4 — Install Dependencies

In [ ]:
!pip install -r /content/ctls/requirements.txt -q
print('Dependencies installed.')

## Step 5 — Verify

In [ ]:
import torch

print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')

print(f'\nRepo root contents:')
for item in sorted(os.listdir('/content/ctls')):
    print(f'  {item}')

print('\nAll good — open a stage notebook from /content/ctls/notebooks/')

## Step 6 — Copy Notebooks to Drive (so you can open them normally)

In [ ]:
import shutil

NOTEBOOKS_ON_DRIVE = f'{DRIVE_BASE}/notebooks'
os.makedirs(NOTEBOOKS_ON_DRIVE, exist_ok=True)

stages = [
    'stage1_baseline',
    'stage2_ctls',
    'stage3_embedding_compare',
    'stage4_ablation',
    'stage5_interpretability',
]

for name in stages:
    src = f'{REPO_DIR}/notebooks/{name}.ipynb'
    dst = f'{NOTEBOOKS_ON_DRIVE}/{name}.ipynb'
    shutil.copy2(src, dst)
    print(f'Copied -> {dst}')

print('\nDone. To open a stage notebook:')
print('  File → Open notebook → Google Drive tab')
print(f'  Navigate to: My Drive / ctls / notebooks /')